# Load and check the anndata object

- Dimensions and slots: number of cells/genes, presence of layers, obsm, uns
- Matrix content: type, sparsity, dtype, value range and integer-ness of the main matrix and of the counts layer (raw counts vs transformed)
- Genes (.var): uniqueness of var_name, available columns, gene symbols vs gene_ids
- Metadata (.obs): distributions of response, treatment, cohort, dataset_origin, fraction; cells per batch and sample
- Design consistency: crosstabs cohort × response (for the batch_key choice), cohort × treatment and dataset_origin × cohort, with patients missing a treatment condition or a CD45 fraction
- Missing values: NaNs in the checked metadata fields

## Packages and paths

In [1]:
# Core scverse libraries
from __future__ import annotations
import logging

# Main
import anndata as ad 
import scanpy as sc
import pandas as pd
import numpy as np 
import scipy.sparse as sp
import os

# Set the level of verbosity of scanpy to get more detailed info
sc.settings.verbosity = 3

# Tell Jupyter to show all outputs and not only the last one
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

!pip list 

Package                 Version     Build
----------------------- ----------- -----
absl-py                 2.5.1
aiohappyeyeballs        2.7.1
aiohttp                 3.14.5
aiosignal               1.4.0
anndata                 0.13.4
anndata2ri              2.0
annotated-types         0.8.0
annoy                   1.17.3
array-api-compat        1.15.0
asttokens               3.0.2
attrs                   26.1.0
bbknn                   1.6.0
cached-property         2.0.1
celltypist              1.7.1
certifi                 2026.7.22
cffi                    2.1.1
charset-normalizer      3.5.2
click                   8.5.0
cloudpickle             3.1.2
comm                    0.2.3
contourpy               1.4.0
cycler                  0.12.1
Cython                  3.3.0
debugpy                 1.8.22
Deprecated              3.0.0
docrep                  0.3.2
donfig                  0.8.1.post1
drvi-py                 0.2.7
et_xmlfile              2.0.0
executing               2.3.0
f

Setup the directory of the data. Raw and heavy intermediate objects live here; the repo holds only code.

In [2]:
try:
    SCRIPT_DIR = os.path.dirname(os.path.abspath(__file__))
except NameError:
    SCRIPT_DIR = os.getcwd() # interactive fallback
PHASE_DIR = os.path.dirname(SCRIPT_DIR) # 01_pre_processing/

DATA_DIR = os.environ.get("DATA_DIR")
if not DATA_DIR:
    raise SystemExit("set DATA_DIR first: export DATA_DIR=<data root>")
os.environ["DATA_DIR"] = DATA_DIR

# Input: concatenated object from phase 00_6
INPUT_H5AD = os.path.join(DATA_DIR, "all_samples_combined.h5ad")
print("Input :", INPUT_H5AD, "| exists:", os.path.exists(INPUT_H5AD))

Input : /home/albertoc/Desktop/QCB-Master-Thesis/datasets/all_samples_combined.h5ad | exists: True


## Preliminary exploration of the anndata object
all_samples_combined.h5ad was created in 00_raw_data_processing using CellRanger v 4.0.0.

### Matrix check

In [3]:
# Load the object (this is the only slow step; depends on file size)
adata = ad.read_h5ad(INPUT_H5AD)
adata

AnnData object with n_obs × n_vars = 743336 × 33538
    obs: 'sample', 'batch', 'cohort', 'treatment', 'fraction', 'dataset_origin', 'response'
    var: 'gene_ids', 'feature_types', 'genome'
    layers: 'counts', None (.X)

In [4]:
# Check the matrix of the object
df = pd.DataFrame(
    adata.X[:50].toarray(),
    index=adata.obs_names[:50], # cell barcodes as row labels
    columns=adata.var_names # gene symbols as column labels
)
df.iloc[:, :50] # first 50 cells, first 50 genes

,MIR1302-2HG,FAM138A,OR4F5,AL627309.1,AL627309.3,AL627309.2,AL627309.4,AL732372.1,OR4F29,AC114498.1,...,B3GALT6,C1QTNF12,AL162741.1,UBE2J2,LINC01786,SCNN1D,ACAP3,PUSL1,INTS11,CPTP
P01_A_P_AAACCTGAGAGTAATC-1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
P01_A_P_AAACCTGAGATGTGTA-1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
P01_A_P_AAACCTGAGCGCTTAT-1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
P01_A_P_AAACCTGAGCTCTCGG-1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
P01_A_P_AAACCTGAGGCATGTG-1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
P01_A_P_AAACCTGAGGCCCGTT-1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
P01_A_P_AAACCTGAGTAATCCC-1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
P01_A_P_AAACCTGAGTCTCCTC-1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
P01_A_P_AAACCTGAGTGTACGG-1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
P01_A_P_AAACCTGCAAACCCAT-1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [5]:
# General dimensions
n_cells, n_genes = adata.shape
print(f"n_cells (obs) : {n_cells:,}")
print(f"n_genes (var) : {n_genes:,}")
print(f"Layers present: {list(adata.layers.keys())}")
print(f"obsm keys     : {list(adata.obsm.keys())}") # absent
print(f"uns keys      : {list(adata.uns.keys())}") # absent

n_cells (obs) : 743,336
n_genes (var) : 33,538
Layers present: ['counts', None]
obsm keys     : []
uns keys      : []


In [6]:
adata.obs["sample"].head
adata.obs["batch"].head

<bound method NDFrame.head of P01_A_P_AAACCTGAGAGTAATC-1    P01_A_P
P01_A_P_AAACCTGAGATGTGTA-1    P01_A_P
P01_A_P_AAACCTGAGCGCTTAT-1    P01_A_P
P01_A_P_AAACCTGAGCTCTCGG-1    P01_A_P
P01_A_P_AAACCTGAGGCATGTG-1    P01_A_P
                               ...   
P66_C_P_TTCTCCTAGTTAGGTA-1    P66_C_P
P66_C_P_TTGGAACAGAGCTATA-1    P66_C_P
P66_C_P_TTTATGCGTTGTGGCC-1    P66_C_P
P66_C_P_TTTGTCACACAGAGGT-1    P66_C_P
P66_C_P_TTTGTCAGTGCGGTAA-1    P66_C_P
Name: sample, Length: 743336, dtype: category
Categories (149, str): ['P01_A_P', 'P01_C_P', 'P02_A_P', 'P02_B_P', ..., 'P66_B_N', 'P66_B_P', 'P66_C_N', 'P66_C_P']>

<bound method NDFrame.head of P01_A_P_AAACCTGAGAGTAATC-1    h01A_P
P01_A_P_AAACCTGAGATGTGTA-1    h01A_P
P01_A_P_AAACCTGAGCGCTTAT-1    h01A_P
P01_A_P_AAACCTGAGCTCTCGG-1    h01A_P
P01_A_P_AAACCTGAGGCATGTG-1    h01A_P
                               ...  
P66_C_P_TTCTCCTAGTTAGGTA-1    h66C_P
P66_C_P_TTGGAACAGAGCTATA-1    h66C_P
P66_C_P_TTTATGCGTTGTGGCC-1    h66C_P
P66_C_P_TTTGTCACACAGAGGT-1    h66C_P
P66_C_P_TTTGTCAGTGCGGTAA-1    h66C_P
Name: batch, Length: 743336, dtype: category
Categories (149, str): ['h01A_P', 'h01C_P', 'h02A_P', 'h02B_P', ..., 'h66B_N', 'h66B_P', 'h66C_N', 'h66C_P']>

In [7]:
# Raw data check
def matrix_report(mat, name):
    """Report dtype, sparsity, value range, and integer-ness of a matrix (first 1000 cells)."""
    is_sparse = sp.issparse(mat)
    print(f"[{name}]")
    print(f"  type      : {type(mat).__name__}  (sparse={is_sparse})")
    print(f"  dtype     : {mat.dtype}")

    sample = mat[:1000].toarray() if is_sparse else np.asarray(mat[:1000])
    print(f"  min / max : {float(sample.min()):.4f} / {float(sample.max()):.4f}   (first 1000 cells)")

    vals = np.asarray(sample).ravel()
    all_integer = np.allclose(vals, np.round(vals))
    verdict = "looks like RAW COUNTS" if all_integer else "looks TRANSFORMED (log/norm)"
    print(f"  integer?  : {all_integer}   -> {verdict}")

matrix_report(adata.X, ".X")

[.X]
  type      : csr_matrix  (sparse=True)
  dtype     : float32
  min / max : 0.0000 / 5733.0000   (first 1000 cells)
  integer?  : True   -> looks like RAW COUNTS


In [8]:
# Report the counts layer if present
has_counts = "counts" in adata.layers
print(f"'counts' layer present: {'YES' if has_counts else 'MISSING'}\n")

if has_counts:
    matrix_report(adata.layers["counts"], ".layers['counts']")

'counts' layer present: YES

[.layers['counts']]
  type      : csr_matrix  (sparse=True)
  dtype     : float32
  min / max : 0.0000 / 5733.0000   (first 1000 cells)
  integer?  : True   -> looks like RAW COUNTS


In [9]:
print(f"var_names unique : {adata.var_names.is_unique}")
print(f"var columns      : {adata.var.columns.tolist()}")
print("First 5 var_names:", adata.var_names[:5].tolist())
if "gene_ids" in adata.var.columns:
    print("First 5 gene_ids :", adata.var['gene_ids'][:5].tolist())

var_names unique : True
var columns      : ['gene_ids', 'feature_types', 'genome']
First 5 var_names: ['MIR1302-2HG', 'FAM138A', 'OR4F5', 'AL627309.1', 'AL627309.3']
First 5 gene_ids : ['ENSG00000243485', 'ENSG00000237613', 'ENSG00000186092', 'ENSG00000238009', 'ENSG00000239945']


### Metadata (.obs) check

In [10]:
print("All obs columns:")
print(adata.obs.columns.tolist())

All obs columns:
['sample', 'batch', 'cohort', 'treatment', 'fraction', 'dataset_origin', 'response']


In [11]:
# Distributions of the biologically/technically meaningful low-cardinality fields
meta_fields = ["response", "treatment", "cohort", "dataset_origin", "fraction"]

for field in meta_fields:
    print("-" * 40)
    if field not in adata.obs.columns:
        print(f"[{field}] NOT PRESENT in .obs")
        continue
    print(f"[{field}]  (n unique = {adata.obs[field].nunique()})")
    print(adata.obs[field].value_counts(dropna=False))
    print()

----------------------------------------
[response]  (n unique = 3)
response
R2    289981
R1    230954
NR    222401
Name: count, dtype: int64

----------------------------------------
[treatment]  (n unique = 3)
treatment
PD1      305033
BASE     234937
RTPD1    203366
Name: count, dtype: int64

----------------------------------------
[cohort]  (n unique = 34)
cohort
Patient53    53153
Patient16    47229
Patient43    42321
Patient56    41911
Patient09    38306
Patient18    35725
Patient12    31987
Patient63    29529
Patient02    28727
Patient19    28267
Patient01    26716
Patient47    25092
Patient10    24282
Patient26    24221
Patient57    23742
Patient52    22075
Patient17    19500
Patient46    19066
Patient06    17929
Patient05    17387
Patient42    16759
Patient11    15142
Patient03    14467
Patient45    12824
Patient20    12452
Patient39    11591
Patient15    10393
Patient23    10327
Patient07     9440
Patient30     8792
Patient04     8675
Patient48     6882
Patient66     5736
Pa

In [12]:
# High-cardinality fields (per-library): summarize cells-per-group
high_card_fields = ["batch", "sample"]

for field in high_card_fields:
    print("-" * 40)
    if field not in adata.obs.columns:
        print(f"[{field}] NOT PRESENT in .obs")
        continue
    vc = adata.obs[field].value_counts(dropna=False)
    print(f"[{field}]  (n unique = {adata.obs[field].nunique()})")
    print(f"  cells per group: min={vc.min()}, median={int(vc.median())}, "
          f"max={vc.max()}, mean={vc.mean():.1f}")
    print()

----------------------------------------
[batch]  (n unique = 149)
  cells per group: min=109, median=3950, max=19289, mean=4988.8

----------------------------------------
[sample]  (n unique = 149)
  cells per group: min=109, median=3950, max=19289, mean=4988.8



Patients 01 and 30 have 0 cells in PD1 and RTPD1 respectively, they will be removed in section 03.

The metadata column fraction will be modified after the annotation in 01_4_cc_and_annotation/fraction_reassignment.py

In [ ]:
# Design crosstabs: cohort x response, cohort x treatment, CD45 fraction x cohort

def safe_crosstab(a, b):
    if a in adata.obs.columns and b in adata.obs.columns:
        print(f"{a}  x  {b}:")
        display(pd.crosstab(adata.obs[a], adata.obs[b]))
    else:
        print(f"{a} x {b}: one or both columns missing")

safe_crosstab("cohort", "response")

safe_crosstab("cohort", "treatment")
# Print patients that have 0 cells in at least one treatment condition
pt_treat = pd.crosstab(adata.obs['cohort'], adata.obs['treatment'])
missing_treat = pt_treat[(pt_treat == 0).any(axis=1)]
print(f"Patients with 0 cells in >=1 treatment condition: {len(missing_treat)} / {pt_treat.shape[0]}")
display(missing_treat)

safe_crosstab("dataset_origin", "cohort")
# Print patients that have 0 cells in one of the two CD45 fractions
pt_frac = pd.crosstab(adata.obs['cohort'], adata.obs['dataset_origin'])
missing_frac = pt_frac[(pt_frac == 0).any(axis=1)]
print(f"Patients missing one CD45 fraction: {len(missing_frac)} / {pt_frac.shape[0]}")
display(missing_frac)

cohort  x  response:


response,NR,R1,R2
cohort,,,
Patient01,0,0,26716
Patient02,0,28727,0
Patient03,0,0,14467
Patient04,0,8675,0
Patient05,0,17387,0
Patient06,17929,0,0
Patient07,0,9440,0
Patient09,0,38306,0
Patient10,24282,0,0


cohort  x  treatment:


treatment,BASE,PD1,RTPD1
cohort,,,
Patient01,11105,0,15611
Patient02,11054,12604,5069
Patient03,4832,7182,2453
Patient04,3816,1451,3408
Patient05,7905,5744,3738
Patient06,9688,2471,5770
Patient07,3730,618,5092
Patient09,19289,11530,7487
Patient10,7917,6450,9915


Patients with 0 cells in >=1 treatment condition: 2 / 34


treatment,BASE,PD1,RTPD1
cohort,,,
Patient01,11105,0,15611
Patient30,311,8481,0


dataset_origin  x  cohort:


cohort,Patient01,Patient02,Patient03,Patient04,Patient05,Patient06,Patient07,Patient09,Patient10,Patient11,...,Patient46,Patient47,Patient48,Patient52,Patient53,Patient56,Patient57,Patient63,Patient64,Patient66
dataset_origin,,,,,,,,,,,,,,,,,,,,,
immune,26716,28727,14467,8675,17387,17929,9440,38306,24282,15142,...,12267,12602,2574,8082,25490,25225,20684,16840,1509,5026
non_immune,0,0,0,0,0,0,0,0,0,0,...,6799,12490,4308,13993,27663,16686,3058,12689,1182,710


Patients missing one CD45 fraction: 18 / 34


dataset_origin,immune,non_immune
cohort,,
Patient01,26716,0
Patient02,28727,0
Patient03,14467,0
Patient04,8675,0
Patient05,17387,0
Patient06,17929,0
Patient07,9440,0
Patient09,38306,0
Patient10,24282,0


In [14]:
check_cols = meta_fields + [f for f in high_card_fields if f in adata.obs.columns]
na_counts = adata.obs[check_cols].isna().sum()
if (na_counts > 0).any():
    print(na_counts[na_counts > 0])
else:
    print("No NaNs in the checked metadata fields.")

No NaNs in the checked metadata fields.
